# 1. Environment Setup for Playwright

Install required Python libraries and browser dependencies for Playwright scraping in Google Colab.

In [ ]:
!pip install pytest-playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 15.8 MB/s eta 0:00:00


In [ ]:
!playwright install

175.4 MiB [] 0% 361.4s175.4 MiB [] 0% 67.8s175.4 MiB [] 0% 41.3s175.4 MiB [] 0% 28.7s175.4 MiB [] 0% 26.9s175.4 MiB [] 0% 21.9s175.4 MiB [] 0% 13.1s175.4 MiB [] 1% 9.0s175.4 MiB [] 2% 7.2s175.4 MiB [] 2% 8.0s175.4 MiB [] 2% 8.9s175.4 MiB [] 2% 8.8s175.4 MiB [] 2% 9.1s175.4 MiB [] 2% 9.2s175.4 MiB [] 3% 8.3s175.4 MiB [] 3% 7.3s175.4 MiB [] 4% 6.8s175.4 MiB [] 5% 6.1s175.4 MiB [] 5% 5.6s175.4 MiB [] 6% 5.6s175.4 MiB [] 6% 5.1s175.4 MiB [] 7% 5.0s175.4 MiB [] 7% 4.9s175.4 MiB [] 7% 5.3s175.4 MiB [] 8% 5.0s175.4 MiB [] 9% 5.1s175.4 MiB [] 9% 5.4s175.4 MiB [] 9% 5.2s175.4 MiB [] 10% 5.2s175.4 MiB [] 10% 5.1s175.4 MiB [] 11% 5.0s175.4 MiB [] 12% 5.0s175.4 MiB [] 13% 4.8s175.4 MiB [] 13% 4.7s175.4 MiB [] 14% 4.7s175.4 MiB [] 14% 4.6s175.4 MiB [] 15% 4.6s175.4 MiB [] 16% 4.5s175.4 MiB [] 16% 4.4s175.4 MiB [] 17% 4.4s175.4 MiB [] 17% 4.5s175.4 MiB [] 18% 4.5s175.4 MiB [] 19% 4.4s175.4 MiB [] 19% 4.3s175.4 MiB [] 20% 4.3s175.4 MiB [] 21% 4.2s175.4 MiB [] 21% 4.3s175.4 MiB [] 21% 4.4s175.4 MiB []

In [ ]:
!apt-get update
!apt-get install -y \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpangocairo-1.0-0 \
    libpango-1.0-0 \
    libcairo2 \
    libgtk-3-0

!playwright install chromium

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 https://cli.github.com/packages stable InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,947 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,070 kB]


# 2. Import Libraries and Configure Async Support

Import required modules and enable asynchronous execution inside Colab notebooks.

In [ ]:
import nest_asyncio
import asyncio
import re
import csv
import os
import json
import time as time_module

from datetime import datetime
from playwright.async_api import async_playwright

nest_asyncio.apply()

# 3. Google Drive Storage Setup

Mount Google Drive and define helper functions for saving scraper outputs and progress files.

In [ ]:
# =========================================
# GOOGLE DRIVE SETUP
# =========================================

USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/malay_mail_scraper"

def setup_drive():
    """Mount Google Drive and create output folder if needed."""

    if not USE_DRIVE:
        return

    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)

        os.makedirs(DRIVE_DIR, exist_ok=True)

        print(f"✅ Google Drive mounted — files saved to: {DRIVE_DIR}")

    except Exception as e:
        print(f"⚠️ Could not mount Google Drive: {e}")
        print("Files will be saved locally instead.")

def get_path(filename):
    """Return full save path."""

    if USE_DRIVE:
        return os.path.join(DRIVE_DIR, filename)

    return filename

# 4. Resume Progress Management

Save and reload scraper progress to allow interrupted scraping sessions to resume automatically.

In [ ]:
# =========================================
# PROGRESS FILES
# =========================================

def progress_file(category):
    return get_path(f"malay_mail_{category}_progress.json")


def load_progress(category):

    pfile = progress_file(category)
    csvfile = get_path(f"malay_mail_{category}.csv")

    existing_data = []

    if os.path.exists(csvfile) and os.path.getsize(csvfile) > 0:

        try:
            with open(csvfile, newline="", encoding="utf-8") as f:
                existing_data = list(csv.DictReader(f))

            fixed = 0

            for row in existing_data:

                if not row.get("id"):

                    m = re.search(r"/(\d+)/?$", row.get("link", ""))

                    if m:
                        row["id"] = m.group(1)
                        fixed += 1

            if fixed:
                print(f"🔧 Backfilled {fixed} missing IDs")

            print(f"📂 Loaded {len(existing_data)} rows")

        except Exception as e:
            print(f"⚠️ Could not read CSV: {e}")

    if os.path.exists(pfile):

        try:
            with open(pfile, encoding="utf-8") as f:
                state = json.load(f)

            current_page = state.get("current_page", 1)
            seen_urls = set(state.get("seen_urls", []))
            seen_pages = set(tuple(p) for p in state.get("seen_pages", []))

            print(f"▶️ Resuming from page {current_page}")

            return current_page, seen_urls, seen_pages, existing_data

        except Exception as e:
            print(f"⚠️ Could not read progress file: {e}")

    print("🆕 Starting fresh")

    return 1, set(), set(), existing_data


def save_progress(category, current_page, seen_urls, seen_pages, total_scraped):

    pfile = progress_file(category)

    state = {
        "current_page": current_page,
        "total_scraped": total_scraped,
        "seen_urls": list(seen_urls),
        "seen_pages": [list(p) for p in seen_pages],
    }

    with open(pfile, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


def clear_progress(category):

    pfile = progress_file(category)

    if os.path.exists(pfile):
        os.remove(pfile)

        print(f"🗑️ Removed progress file")

# 5. CSV Export Functions

Save scraped article data into structured CSV files with checkpoint support.

In [ ]:
# =========================================
# CSV EXPORT
# =========================================

def save_to_csv(data, category, is_checkpoint=False):

    if not data:
        print("⚠️ No data to save.")
        return False

    filename = get_path(f"malay_mail_{category}.csv")

    FIELDNAMES = [
        "id",
        "title",
        "category",
        "day",
        "date",
        "time",
        "author",
        "content",
        "link"
    ]

    try:

        with open(filename, "w", newline="", encoding="utf-8-sig") as f:

            writer = csv.DictWriter(
                f,
                fieldnames=FIELDNAMES,
                extrasaction="ignore"
            )

            writer.writeheader()
            writer.writerows(data)

        print(f"✅ Saved {len(data)} rows")

        return True

    except Exception as e:

        print(f"❌ Save failed: {e}")

        return False

# 6. Helper Utility Functions

Utility functions for formatting dates, cleaning titles, and displaying elapsed scraping time.

In [ ]:
# =========================================
# HELPER FUNCTIONS
# =========================================

def parse_date_time(raw):

    if not raw or raw == "N/A":
        return "N/A", "N/A", "N/A"

    clean_iso = re.sub(r"[+-]\d{2}:\d{2}$", "", raw).strip()

    iso_specs = [
        ("%Y-%m-%dT%H:%M:%S", 19),
        ("%Y-%m-%d %H:%M:%S", 19),
        ("%Y-%m-%dT%H:%M", 16),
        ("%Y-%m-%d %H:%M", 16),
    ]

    for fmt, char_len in iso_specs:

        try:
            dt = datetime.strptime(clean_iso[:char_len], fmt)

            day_out = dt.strftime("%A")
            date_out = dt.strftime("%-d %B %Y")
            time_out = dt.strftime("%I:%M %p").lstrip("0") + " MYT"

            return day_out, date_out, time_out

        except Exception:
            pass

    return "N/A", raw.strip(), "N/A"


def clean_title(raw_title):

    return re.sub(r"\s+", " ", raw_title).strip()


def fmt_elapsed(seconds):

    seconds = int(seconds)

    if seconds < 60:
        return f"{seconds}s"

    return f"{seconds // 60}m {seconds % 60}s"

# 7. Result Display Function

Display scraped article information in a readable console format.

In [ ]:
# =========================================
# PRINT RESULTS
# =========================================

def print_results(data, category, elapsed):

    print("=" * 70)

    print(f"SCRAPE RESULTS — {len(data)} article(s)")
    print(f"Category: {category.upper()}")
    print(f"Time: {elapsed}")

    print("=" * 70)

    for i, row in enumerate(data, 1):

        print(f"\n[{i}] {row['title']}")
        print(f"ID: {row['id']}")
        print(f"Date: {row['date']}")
        print(f"Author: {row['author']}")
        print(f"Link: {row['link']}")

# 8. Main Malay Mail Scraper

Core Playwright scraper that navigates article pages, extracts article data, handles deduplication, and saves progress automatically.

In [ ]:
# =========================================
# MAIN SCRAPER
# =========================================

async def scrape_malay_mail(
    category,
    target_goal,
    checkpoint_every=50,
    extend=False,
    start_page=1,
):

    setup_drive()

    start_time = time_module.time()

    category = category.lower()

    base_url = f"https://www.malaymail.com/morearticles/{category}"

    current_page, seen_urls, seen_pages, all_data = load_progress(category)

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-setuid-sandbox",
            ]
        )

        context = await browser.new_context()

        page = await context.new_page()

        while len(all_data) < target_goal:

            url = f"{base_url}?pgno={current_page}"

            print(f"\n🌐 Page {current_page}")

            try:
                await page.goto(
                    url,
                    wait_until="domcontentloaded",
                    timeout=60000
                )

            except Exception as e:

                print(f"⚠️ Failed page load: {e}")

                break

            await asyncio.sleep(5)

            articles = await page.query_selector_all(".article-item")

            if not articles:

                print("📄 No more articles")

                break

            for article in articles:

                try:

                    title_el = await article.query_selector("h2 a, h3 a")

                    if not title_el:
                        continue

                    title = clean_title(await title_el.inner_text())

                    link = await title_el.get_attribute("href")

                    if not link:
                        continue

                    if link.startswith("/"):
                        link = "https://www.malaymail.com" + link

                    if link in seen_urls:
                        continue

                    seen_urls.add(link)

                    row = {
                        "title": title,
                        "link": link,
                    }

                    all_data.append(row)

                    print(f"✅ {title}")

                except Exception as e:

                    print(f"⚠️ Error: {e}")

            current_page += 1

        save_to_csv(all_data, category)

        await browser.close()

# 9. Run the Scraper

Start the scraping process by specifying category, target article count, and checkpoint settings.

In [ ]:
await scrape_malay_mail(
    category="malaysia",
    target_goal=15000,
    checkpoint_every=50,
    start_page=1,
    extend=True,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted — files saved to: /content/drive/MyDrive/malay_mail_scraper
🔧 Backfilled 5910 missing IDs
📂 Loaded 5910 rows
▶️ Resuming from page 388
✅ Saved 5910 rows


Since the scraping process took a long execution time due to the large volume of articles. 9 categories were distributed among members:

1. Ru Qian: malaysia,singapore,money
2. Yi Ya: world,life,eat/drink
3. Adriana: showbiz,sports,tech/gadgets